# 04 - Models: Decision Tree con TF-IDF + SVD

Este notebook entrena un modelo de la familia `04_models_` usando **Decision Tree** sobre la codificacion **TF-IDF** generada en `02_encoding.ipynb`.

Al igual que Random Forest, los arboles de decision no escalan bien sobre matrices TF-IDF de muy alta dimension, por lo que se usa **TruncatedSVD** para reducir dimensionalidad antes del entrenamiento. El flujo mantiene el mismo patron de evaluacion y registro en **MLflow** del notebook `03_baseline.ipynb`.

## 1. Instalacion e imports

In [1]:
!pip install -q scikit-learn scipy joblib matplotlib seaborn mlflow

In [2]:
import os
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import load_npz, vstack
from sklearn.decomposition import TruncatedSVD
from sklearn.tree import DecisionTreeClassifier          # ← Decision Tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

SEED       = 42
MEMBER     = "Alan Osorio"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"
EXPERIMENT = "elongacion"   # ← ajusta al nombre de tus artefactos

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
np.random.seed(SEED)
print("OK - Imports listos. SEED:", SEED)

OK - Imports listos. SEED: 42


## 2. Carga de artefactos

Se reutilizan las matrices sparse TF-IDF y labels generadas en `02_encoding.ipynb`.

In [3]:
os.chdir('/home/ec2-user/SageMaker/Sin norm. alarg.')

X_tr          = load_npz(f'X_tr_{EXPERIMENT}.npz')
X_val         = load_npz(f'X_val_{EXPERIMENT}.npz')
X_train_tfidf = load_npz(f'X_train_tfidf_{EXPERIMENT}.npz')
X_test_tfidf  = load_npz(f'X_test_tfidf_{EXPERIMENT}.npz')

y_tr    = joblib.load(f'y_tr_{EXPERIMENT}.pkl')
y_val   = joblib.load(f'y_val_{EXPERIMENT}.pkl')
y_train = joblib.load(f'y_train_{EXPERIMENT}.pkl')
y_test  = joblib.load(f'y_test_{EXPERIMENT}.pkl')

print('Shapes cargados:')
print('  X_tr          :', X_tr.shape)
print('  X_val         :', X_val.shape)
print('  X_train_tfidf :', X_train_tfidf.shape)
print('  X_test_tfidf  :', X_test_tfidf.shape)
print('  y_tr          :', len(y_tr))
print('  y_val         :', len(y_val))
print('  y_train       :', len(y_train))
print('  y_test        :', len(y_test))

Shapes cargados:
  X_tr          : (1088000, 46635)
  X_val         : (272000, 46635)
  X_train_tfidf : (1360000, 46635)
  X_test_tfidf  : (240000, 46635)
  y_tr          : 1088000
  y_val         : 272000
  y_train       : 1360000
  y_test        : 240000


## 3. Muestreo para entrenamiento del Decision Tree

Un `DecisionTreeClassifier` sobre 46k atributos TF-IDF y 1.36M registros es costoso en memoria y tiempo. Se usa una **muestra estratificada** del train/validation y se reduce dimensionalidad con SVD para que el experimento sea ejecutable y reproducible.

In [4]:
# Dataset completo — sin muestreo (igual que antes)
X_tr_sample,  y_tr_sample  = X_tr,  np.asarray(y_tr)
X_val_sample, y_val_sample = X_val, np.asarray(y_val)

print('Datos completos para tuning:')
print(' X_tr_sample :', X_tr_sample.shape)   # (1_088_000, 46_635)
print(' X_val_sample:', X_val_sample.shape)  # (272_000,   46_635)
print(' balance y_tr_sample :', np.bincount(y_tr_sample))
print(' balance y_val_sample:', np.bincount(y_val_sample))


Datos completos para tuning:
  X_tr_sample : (1088000, 46635)
  X_val_sample: (272000, 46635)
  balance y_tr_sample : [543897 544103]
  balance y_val_sample: [135974 136026]


## 4. Busqueda de hiperparametros

Se entrena un pipeline `TruncatedSVD → DecisionTreeClassifier` y se selecciona la mejor configuracion por `f1_macro` en validacion.

### Hiperparametros explorados
| Parametro | Descripcion |
|---|---|
| `n_components` | Dimensiones SVD antes del arbol |
| `max_depth` | Profundidad maxima del arbol — controla overfitting |
| `min_samples_leaf` | Minimo de muestras en hoja — regularizacion |
| `criterion` | Funcion de impureza: `gini` o `entropy` |

In [ ]:
import time
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

# ─────────────────────────────────────────────────────────────────────────────
# PASO 1 — Muestra estratificada ANTES del SVD
# Reducimos filas PRIMERO (1.08M → 108k) y luego aplicamos SVD sobre la muestra.
# Así el SVD opera sobre 108k × 46k en lugar de 1.08M × 46k → ~10x más rápido.
#
# ¿Es válido? Sí: el objetivo del SVD aquí es reducir dimensiones para el tuning.
# El SVD final (celda de entrenamiento) se fiteará sobre el dataset completo.
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_RATIO   = 0.10    # 10% para tuning → ~108k filas
MAX_COMPONENTS = 300     # Máximo n_components del param_grid

# ── 1. Muestra estratificada del train ───────────────────────────────────────
y_tr_arr = np.asarray(y_tr)
y_val_arr = np.asarray(y_val)

sss = StratifiedShuffleSplit(n_splits=1, train_size=SAMPLE_RATIO, random_state=SEED)
idx_tune, _ = next(sss.split(X_tr, y_tr_arr))  # X_tr es la matriz sparse original

X_tr_tune_sparse = X_tr[idx_tune]   # (108_800, 46_635) sparse — muestra
y_tr_tune        = y_tr_arr[idx_tune]

print(f"Muestra de tuning : {X_tr_tune_sparse.shape}")
print(f"Balance clases    : {np.bincount(y_tr_tune)}")

# ── 2. SVD sobre la muestra (~108k filas) — ahora sí rápido ──────────────────
print(f"\n⏳ TruncatedSVD ({MAX_COMPONENTS} comps) sobre muestra de tuning...")
t0 = time.time()

svd_tune = TruncatedSVD(
    n_components=MAX_COMPONENTS,
    algorithm='randomized',
    n_iter=2,
    random_state=SEED,
)
X_tr_tune_svd = svd_tune.fit_transform(X_tr_tune_sparse)   # (108_800, 300) denso
X_val_svd     = svd_tune.transform(X_val)                  # (272_000, 300) denso — val completo

print(f"✅ SVD listo en {time.time() - t0:.1f}s")
print(f"   Varianza explicada ({MAX_COMPONENTS} comps): "
      f"{svd_tune.explained_variance_ratio_.sum():.3f}")

# ── 3. Grid search sobre el DecisionTree ─────────────────────────────────────
param_grid = [
    {'n_components': 200, 'max_depth': 10, 'min_samples_leaf': 10, 'criterion': 'gini'},
    {'n_components': 200, 'max_depth': 20, 'min_samples_leaf': 5,  'criterion': 'gini'},
    {'n_components': 300, 'max_depth': 20, 'min_samples_leaf': 5,  'criterion': 'entropy'},
    {'n_components': 300, 'max_depth': 30, 'min_samples_leaf': 2,  'criterion': 'gini'},
]

results_dt = []
best_dt     = None
best_params = None
best_f1     = -1

print(f"\n{'─'*70}")
for i, params in enumerate(param_grid):
    t_iter = time.time()
    k = params['n_components']

    X_tr_k  = X_tr_tune_svd[:, :k]   # muestra de tuning → fit
    X_val_k = X_val_svd[:,    :k]     # validación completa → métrica real

    dt = DecisionTreeClassifier(
        max_depth=params['max_depth'],
        min_samples_leaf=params['min_samples_leaf'],
        criterion=params['criterion'],
        class_weight='balanced',
        random_state=SEED,
    )
    dt.fit(X_tr_k, y_tr_tune)

    y_val_pred  = dt.predict(X_val_k)
    y_val_proba = dt.predict_proba(X_val_k)[:, 1]

    metrics_row = {
        **params,
        'accuracy':        round(accuracy_score(y_val_arr,  y_val_pred),                  4),
        'f1_macro':        round(f1_score(y_val_arr,        y_val_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_val_arr, y_val_pred, average='macro'), 4),
        'recall_macro':    round(recall_score(y_val_arr,    y_val_pred, average='macro'), 4),
        'roc_auc':         round(roc_auc_score(y_val_arr,   y_val_proba),                 4),
    }
    results_dt.append(metrics_row)

    if metrics_row['f1_macro'] > best_f1:
        best_f1     = metrics_row['f1_macro']
        best_params = params
        best_dt     = dt

    print(f"  [{i+1}/{len(param_grid)}] n_comp={k} | depth={params['max_depth']:2d} "
          f"| leaf={params['min_samples_leaf']:2d} | {params['criterion']:7s} "
          f"→ f1={metrics_row['f1_macro']:.4f}  ({time.time()-t_iter:.1f}s)")

print(f"{'─'*70}")
df_dt = pd.DataFrame(results_dt).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df_dt)
print(f'\n🏆 Mejores hiperparámetros : {best_params}')
print(f'   Mejor f1_macro (val)    : {best_f1:.4f}')
print(f'\n⚠️  Tuning con {SAMPLE_RATIO*100:.0f}% del train. '
      f'Modelo final se reentrena con dataset COMPLETO en la celda siguiente.')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = df_dt.copy()
plot_df['config'] = [
    f"SVD={r.n_components} | depth={r.max_depth} | {r.criterion}"
    for r in plot_df.itertuples()
]
ax.bar(plot_df['config'], plot_df['f1_macro'], color='steelblue')
ax.set_title('Decision Tree - F1 macro en validacion')
ax.set_xlabel('Configuracion')
ax.set_ylabel('F1 macro')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 5. Entrenamiento final

Se reentrena el mejor pipeline usando la union de la muestra de train y validacion.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Entrenamiento final sobre el dataset COMPLETO (train + val unidos = 1.36M)
# Se vuelve a fitear SVD porque ahora el input es X_train_tfidf, no X_tr.
# ─────────────────────────────────────────────────────────────────────────────

X_train_final = X_train_tfidf        # (1_360_000, 46_635)
y_train_final = np.asarray(y_train)

k_best = best_params['n_components']

print(f"⏳ Ajustando SVD final con {k_best} componentes sobre {X_train_final.shape}...")
t0 = time.time()

svd_final = TruncatedSVD(
    n_components=k_best,
    algorithm='randomized',
    n_iter=2,
    random_state=SEED,
)
X_train_final_svd = svd_final.fit_transform(X_train_final)  # (1_360_000, k_best)
print(f"✅ SVD final listo en {time.time() - t0:.1f}s")

# Entrenamos el árbol con los mejores hiperparámetros
model_final_dt = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    criterion=best_params['criterion'],
    class_weight='balanced',
    random_state=SEED,
)
model_final_dt.fit(X_train_final_svd, y_train_final)

# Re-empaquetamos como Pipeline para mantener compatibilidad con el resto del notebook
# (predicción en test, MLflow, joblib.dump, etc.)
model_final = Pipeline([
    ('svd', svd_final),
    ('dt',  model_final_dt),
])

print('✅ Modelo final entrenado.')
print(f'   Train shape  : {X_train_final.shape}')
print(f'   n_components : {k_best}')
print(f'   max_depth    : {best_params["max_depth"]}')
print(f'   criterion    : {best_params["criterion"]}')


## 6. Evaluacion final en test

In [ ]:
y_pred  = model_final.predict(X_test_tfidf)
y_proba = model_final.predict_proba(X_test_tfidf)[:, 1]

acc       = accuracy_score(y_test,  y_pred)
f1        = f1_score(y_test,        y_pred, average='macro')
auc       = roc_auc_score(y_test,   y_proba)
precision = precision_score(y_test, y_pred, average='macro')
recall    = recall_score(y_test,    y_pred, average='macro')

print('Metricas en test:')
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 macro  : {f1:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  ROC AUC   : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo (0)', 'Positivo (1)'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Decision Tree + TF-IDF/SVD')
plt.tight_layout()
plt.show()

## 7. Guardar modelo y resultados

In [ ]:
joblib.dump(model_final, 'dt_tfidf_svd_model.pkl')

results_final = pd.DataFrame([{
    'modelo':             'DecisionTreeClassifier',
    'encoding':           'TF-IDF + TruncatedSVD',
    'member':             MEMBER,
    'train_sample_size':  'FULL',                      # ← antes era el tamaño de muestra
    'val_sample_size':    'FULL',                      # ← antes era el tamaño de muestra
    'final_train_size':   len(y_train_final),
    'n_components':       best_params['n_components'],
    'max_depth':          best_params['max_depth'],
    'min_samples_leaf':   best_params['min_samples_leaf'],
    'criterion':          best_params['criterion'],
    'accuracy':           round(acc, 4),
    'f1_macro':           round(f1, 4),
    'roc_auc':            round(auc, 4),
}])

results_final.to_csv('results_dt_tfidf_svd.csv', index=False)

print('Archivos guardados:')
print('  dt_tfidf_svd_model.pkl   -> pipeline SVD + Decision Tree (FULL dataset)')
print('  results_dt_tfidf_svd.csv -> tabla de metricas')
print()
print(results_final.to_string(index=False))


## 8. Registro en MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'DecisionTree_TFIDF-SVD_{MEMBER}') as run:
    mlflow.set_tags({
        'user':       'Alan Osorio',
        'member':     MEMBER,
        'model_type': 'DecisionTreeClassifier',
        'encoding':   'TF-IDF + TruncatedSVD',
        'dataset':    'Sentiment140Twitter',
    })

    mlflow.log_params({
        'prep_remove_urls':      True,
        'prep_remove_mentions':  True,
        'prep_remove_hashtags':  True,
        'prep_remove_emojis':    True,
        'prep_remove_stopwords': False,
        'prep_lemmatization':    False,
    })

    mlflow.log_params({
        'vec_type':         'TfidfVectorizer',
        'vec_max_features': 50000,
        'vec_min_df':       5,
        'vec_max_df':       0.95,
        'vec_ngram_range':  '(1,1)',
        'vec_sublinear_tf': False,
        'vec_norm':         'None',
    })

    mlflow.log_params({
        'model':             'DecisionTreeClassifier',
        'dim_reduction':     'TruncatedSVD',
        'n_components':      best_params['n_components'],
        'max_depth':         best_params['max_depth'],
        'min_samples_leaf':  best_params['min_samples_leaf'],
        'criterion':         best_params['criterion'],
        'class_weight':      'balanced',
        'seed':              SEED,
        'train_sample_size': len(y_tr_sample),
        'val_sample_size':   len(y_val_sample),
        'final_train_size':  len(y_train_final),
        'test_size':         X_test_tfidf.shape[0],
        'vocab_size':        X_train_tfidf.shape[1],
    })

    # Metricas en train
    y_pred_train  = model_final.predict(X_train_final)
    y_proba_train = model_final.predict_proba(X_train_final)[:, 1]
    mlflow.log_metrics({
        'train_accuracy':  round(accuracy_score(y_train_final,  y_pred_train),                    4),
        'train_f1_macro':  round(f1_score(y_train_final,        y_pred_train, average='macro'),   4),
        'train_precision': round(precision_score(y_train_final, y_pred_train, average='macro'),   4),
        'train_recall':    round(recall_score(y_train_final,    y_pred_train, average='macro'),   4),
        'train_roc_auc':   round(roc_auc_score(y_train_final,   y_proba_train),                   4),
    })

    # Metricas en validacion
    y_pred_val_f  = best_model.predict(X_val_sample)
    y_proba_val_f = best_model.predict_proba(X_val_sample)[:, 1]
    mlflow.log_metrics({
        'val_accuracy':  round(accuracy_score(y_val_sample,  y_pred_val_f),                    4),
        'val_f1_macro':  round(f1_score(y_val_sample,        y_pred_val_f, average='macro'),   4),
        'val_precision': round(precision_score(y_val_sample, y_pred_val_f, average='macro'),   4),
        'val_recall':    round(recall_score(y_val_sample,    y_pred_val_f, average='macro'),   4),
        'val_roc_auc':   round(roc_auc_score(y_val_sample,   y_proba_val_f),                   4),
    })

    # Metricas en test
    mlflow.log_metrics({
        'test_accuracy':  round(acc,       4),
        'test_f1_macro':  round(f1,        4),
        'test_precision': round(precision, 4),
        'test_recall':    round(recall,    4),
        'test_roc_auc':   round(auc,       4),
    })

    mlflow.log_artifact('results_dt_tfidf_svd.csv')
    mlflow.sklearn.log_model(
        model_final,
        artifact_path='model',
        registered_model_name=f'DT_TFIDF_SVD_{MEMBER}'
    )

    print('=' * 55)
    print(' Run registrado en MLflow')
    print(f' Servidor    : {MLFLOW_URI}')
    print(' Experimento : Parcial_1_NLP')
    print(f' Corrida     : DecisionTree_TFIDF-SVD_{MEMBER}')
    print(f' Run ID      : {run.info.run_id}')
    print(f' Test F1     : {f1:.4f}')
    print(f' Test AUC    : {auc:.4f}')
    print('=' * 55)